In [1]:
import pandas as pd
import tarfile
import gzip
from io import StringIO
import h3
import os
import glob

### This notebook does a temporal smoothing, calculating the moving average of each h3 hexagon based on the neighbouring hexagon values.


###### at this step all columns and days of the opening and curfew period are preserved
###### Opening: 1 Sept - 31 Oct 2020 (inclusive). Curfew: 11 Nov - 23 Dec 2020 (inclusive)

In [2]:
columns_to_sum = ['traffic', 'remainers', 'loc_home', 'loc_work', 'loc_freq',
                  'sex_female', 'sex_male', 'sex_na', 'arpu_low', 'arpu_mid',
                  'arpu_high', 'arpu_na', 'age_young', 'age_mid', 'age_old',
                  'age_na', 'plan_priv', 'plan_corp', 'plan_roam']

In [3]:
# create a custom mean function
# this function sums the group values, however always divides by 7, as each group has 7 neighbors, but some values could be NA
def custom_mean(group):
    summed_values = group.sum()
    averaged_values = summed_values / 7
    return averaged_values

In [4]:
def spat_moving_average(data: pd.DataFrame, k: int = 1) -> pd.DataFrame:
    records = []
    for raster_id in data['raster_id'].unique():
        neighbors = h3.k_ring(raster_id, k)
        temp = data[data['raster_id'].isin(neighbors)].copy()
        
        moving_average = temp.groupby(["year", "month", "day", "hour"])[columns_to_sum].apply(custom_mean)
        for i in moving_average.reset_index().values.tolist():
            records.append([raster_id] + i)
    return pd.DataFrame.from_records(records, columns=['raster_id', "year", "month", "day", "hour"] + columns_to_sum)

In [5]:
checkpoint_dir = "../data/data_gen/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
# Each period is defined as a list of (tar_file, start_day, end_day)
periods = {
    "open": [
        ("2020-09.csv.gz.tar", 1, 30),
        ("2020-10.csv.gz.tar", 1, 31),
    ],
    "curfew": [
        ("2020-11.csv.gz.tar", 11, 30),
        ("2020-12.csv.gz.tar", 1, 23),
    ],
}

In [ ]:
# Resumable reading + smoothing for the opening and curfew periods ---

# Checkpoint 1: if a period's final pickle already exists, the whole period is skipped.
# Checkpoint 2: each individual day is smoothed and saved separately; if that day's
#                     checkpoint file already exists, the day is skipped
# Once all days for a period are done, they are combined into the final open_smoothed.pkl / curfew_smoothed.pkl.

# This means an interrupted run can simply be # re-executed 

def day_from_filename(filename: str) -> int:
    # expects members named like '2020-09-01.csv.gz' -> returns 1
    date_str = filename.split(".")[0]
    return int(date_str.split("-")[-1])

for period_name, month_specs in periods.items():
    final_path = f"../data/data_gen/{period_name}_smoothed.pkl"
    if os.path.exists(final_path):
        print(f"{period_name}: final output already exists, skipping period.")
        continue

    for target_month, start_day, end_day in month_specs:
        print(f"Processing {period_name} data from: {target_month}")
        with tarfile.open(f"/mnt/common-hdd/raw-sources/tkom-data/{target_month}", "r:*") as tar:
            for day_file in tar.getnames():
                day_num = day_from_filename(day_file)
                if day_num < start_day or day_num > end_day:
                    continue

                checkpoint_path = f"{checkpoint_dir}/{period_name}_{day_file.split('.')[0]}.pkl"
                if os.path.exists(checkpoint_path):
                    print(f"  {day_file}: checkpoint exists, skipping.")
                    continue

                print(f"  Reading and smoothing {day_file}")
                with gzip.open(tar.extractfile(day_file), "rb") as f_in:
                    data = StringIO(str(f_in.read(), "utf-8"))
                    df_day = pd.read_csv(data, index_col=0)
                    df_day["count"] = 1

                df_day_smoothed = spat_moving_average(df_day, 1)
                df_day_smoothed.to_pickle(checkpoint_path)

    # Combine all per-day checkpoints for this period into the final pickle
    checkpoint_files = sorted(glob.glob(f"{checkpoint_dir}/{period_name}_*.pkl"))
    df_period_smoothed = pd.concat([pd.read_pickle(f) for f in checkpoint_files], ignore_index=True)
    df_period_smoothed.to_pickle(final_path)
    print(f"{period_name}: saved final output to {final_path}")

open: final output already exists, skipping period.
Processing curfew data from: 2020-11.csv.gz.tar
  2020-11-11.csv.gz: checkpoint exists, skipping.
  2020-11-12.csv.gz: checkpoint exists, skipping.
  2020-11-13.csv.gz: checkpoint exists, skipping.
  2020-11-14.csv.gz: checkpoint exists, skipping.
  2020-11-15.csv.gz: checkpoint exists, skipping.
  2020-11-16.csv.gz: checkpoint exists, skipping.
  2020-11-17.csv.gz: checkpoint exists, skipping.
  2020-11-18.csv.gz: checkpoint exists, skipping.
  2020-11-19.csv.gz: checkpoint exists, skipping.
  Reading and smoothing 2020-11-20.csv.gz


  Reading and smoothing 2020-11-21.csv.gz
  Reading and smoothing 2020-11-22.csv.gz
  Reading and smoothing 2020-11-23.csv.gz
  Reading and smoothing 2020-11-24.csv.gz
  Reading and smoothing 2020-11-25.csv.gz
  Reading and smoothing 2020-11-26.csv.gz
  Reading and smoothing 2020-11-27.csv.gz
  Reading and smoothing 2020-11-28.csv.gz
  Reading and smoothing 2020-11-29.csv.gz
  Reading and smoothing 2020-11-30.csv.gz
Processing curfew data from: 2020-12.csv.gz.tar
  Reading and smoothing 2020-12-10.csv.gz
  Reading and smoothing 2020-12-11.csv.gz
  Reading and smoothing 2020-12-12.csv.gz
  Reading and smoothing 2020-12-13.csv.gz
  Reading and smoothing 2020-12-14.csv.gz
  Reading and smoothing 2020-12-15.csv.gz
  Reading and smoothing 2020-12-16.csv.gz
  Reading and smoothing 2020-12-17.csv.gz
  Reading and smoothing 2020-12-18.csv.gz
  Reading and smoothing 2020-12-19.csv.gz
  Reading and smoothing 2020-12-1.csv.gz
  Reading and smoothing 2020-12-20.csv.gz
  Reading and smoothing 2020-